In [1]:
print("Feature Engineering for Jigsaw Event VOL001 dataset:")

Feature Engineering for Jigsaw Event VOL001 dataset:


In [3]:
import pandas as pd
df = pd.read_csv("../data/JE001_real_sample.csv")
df.head(10)

,date,tickets_sold,instagram_followers,reel_views,days_advertised,day_of_week,month,student_term_time,weather,rainfall,temperature
0,19/07/26,9,49,674,14,Sunday,July,Summer Break,Sunny,0mm,24C
1,23/07/26,11,57,898,18,Thursday,July,Summer Break,Cloudy,0mm,19C
2,28/07/26,18,73,1535,23,Tuesday,July,Summer Break,Sunny,0mm,25C
3,31/07/26,19,74,1569,26,Friday,July,Summer Break,Sunny,0mm,20C
4,06/08/26,25,76,1593,32,Thursday,August,Summer Break,Cloudy,1mm,15C
5,08/08/26,32,77,1603,34,Saturday,August,Summer Break,Sunny,0mm,20C
6,12/08/26,41,78,1642,38,Wednesday,August,Summer Break,Sunny,0mm,26C
7,17/08/26,53,81,1687,43,Monday,August,Summer Break,Cloudy,1mm,20C
8,21/08/26,98,91,1852,48,Friday,August,Summer Break,Sunny,0mm,21C


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   date                 9 non-null      str  
 1   tickets_sold         9 non-null      int64
 2   instagram_followers  9 non-null      int64
 3   reel_views           9 non-null      int64
 4   days_advertised      9 non-null      int64
 5   day_of_week          9 non-null      str  
 6   month                9 non-null      str  
 7   student_term_time    9 non-null      str  
 8   weather              9 non-null      str  
 9   rainfall             9 non-null      str  
 10  temperature          9 non-null      str  
dtypes: int64(4), str(7)
memory usage: 924.0 bytes


In [6]:
df.describe()

,tickets_sold,instagram_followers,reel_views,days_advertised
count,9.000000,9.000000,9.000000,9.000000
mean,34.000000,72.888889,1450.333333,30.666667
std,27.897132,12.604012,391.592901,11.390786
min,9.000000,49.000000,674.000000,14.000000
25%,18.000000,73.000000,1535.000000,23.000000
50%,25.000000,76.000000,1593.000000,32.000000
75%,41.000000,78.000000,1642.000000,38.000000
max,98.000000,91.000000,1852.000000,48.000000


In [7]:
target = "tickets_sold"

In [8]:
df["reel_views_per_follower"] = (df["reel_views"] / df["instagram_followers"])

In [9]:
df[["instagram_followers", "reel_views", "reel_views_per_follower"]].head(10)

,instagram_followers,reel_views,reel_views_per_follower
0,49,674,13.755102
1,57,898,15.754386
2,73,1535,21.027397
3,74,1569,21.202703
4,76,1593,20.960526
5,77,1603,20.818182
6,78,1642,21.051282
7,81,1687,20.827160
8,91,1852,20.351648


In [10]:
campaign_end_date = "21/8/26"
campaign_start_date = "5/7/26"

In [11]:
df["campaign_duration"] = pd.to_datetime(campaign_end_date, format="%d/%m/%y") - pd.to_datetime(campaign_start_date, format="%d/%m/%y")

In [12]:
df["date"] = pd.to_datetime(df["date"], dayfirst=True)

C:\Users\alexb\AppData\Local\Temp\ipykernel_8004\2148271709.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["date"] = pd.to_datetime(df["date"], dayfirst=True)


In [13]:
df["days_until_event"] = (pd.Timestamp("2026-8-21") - df["date"]).dt.days

In [14]:
df["campaign_progress"] = (df["date"] - pd.Timestamp("2026-7-5")).dt.days / df["campaign_duration"].dt.days

In [15]:
df[["instagram_followers", "reel_views", "reel_views_per_follower", "days_until_event", "campaign_progress"]].head(10)

,instagram_followers,reel_views,reel_views_per_follower,days_until_event,campaign_progress
0,49,674,13.755102,33,0.297872
1,57,898,15.754386,29,0.382979
2,73,1535,21.027397,24,0.489362
3,74,1569,21.202703,21,0.553191
4,76,1593,20.960526,15,0.680851
5,77,1603,20.818182,13,0.723404
6,78,1642,21.051282,9,0.808511
7,81,1687,20.827160,4,0.914894
8,91,1852,20.351648,0,1.000000


In [16]:
df["is_weekend"] = df["day_of_week"].isin(["Saturday", "Sunday"]).astype(int)

In [17]:
df["is_student_holiday"] = (df["student_term_time"] == "Summer Break").astype(int)  

In [18]:
df["temperature_celsius"] = (df["temperature"].str.replace("C", "").astype(float))

In [19]:
df["rainfall_mm"] = (df["rainfall"].str.replace("mm", "").astype(float))

In [20]:
df["has_rain"] = (df["rainfall_mm"] > 0).astype(int)

In [21]:
df["rain_severity"] = pd.cut(df["rainfall_mm"], bins=[-0.01, 0, 2, 10, float("inf")], labels=[0, 1, 2, 3]).astype(int)

In [22]:
df["temperature_severity"] = 0
df.loc[df["temperature_celsius"] < 5, "temperature_severity"] = 2
df.loc[df["temperature_celsius"].between(5,10), "temperature_severity"] = 1
df.loc[df["temperature_celsius"].between(10,24), "temperature_severity"] = 0
df.loc[df["temperature_celsius"].between(25,29), "temperature_severity"] = 1
df.loc[df["temperature_celsius"] >= 30, "temperature_severity"] = 2

In [23]:
weather_severity_map = {"Sunny": 0, "Cloudy": 1}
df["weather_severity"] = (df["weather"].map(weather_severity_map))

In [24]:
df["overall_severity"] = (df["rain_severity"] + df["temperature_severity"] + df["weather_severity"])

In [25]:
df[["instagram_followers", "reel_views", "reel_views_per_follower", "days_until_event", "campaign_progress", "rain_severity", "temperature_severity", "weather_severity","overall_severity"]].head(10)

,instagram_followers,reel_views,reel_views_per_follower,days_until_event,campaign_progress,rain_severity,temperature_severity,weather_severity,overall_severity
0,49,674,13.755102,33,0.297872,0,0,0,0
1,57,898,15.754386,29,0.382979,0,0,1,1
2,73,1535,21.027397,24,0.489362,0,1,0,1
3,74,1569,21.202703,21,0.553191,0,0,0,0
4,76,1593,20.960526,15,0.680851,1,0,1,2
5,77,1603,20.818182,13,0.723404,0,0,0,0
6,78,1642,21.051282,9,0.808511,0,1,0,1
7,81,1687,20.827160,4,0.914894,1,0,1,2
8,91,1852,20.351648,0,1.000000,0,0,0,0


In [26]:
print(df.shape)
print(df.columns.tolist())

(9, 24)
['date', 'tickets_sold', 'instagram_followers', 'reel_views', 'days_advertised', 'day_of_week', 'month', 'student_term_time', 'weather', 'rainfall', 'temperature', 'reel_views_per_follower', 'campaign_duration', 'days_until_event', 'campaign_progress', 'is_weekend', 'is_student_holiday', 'temperature_celsius', 'rainfall_mm', 'has_rain', 'rain_severity', 'temperature_severity', 'weather_severity', 'overall_severity']


In [27]:
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))

print("\nEngineered variables:")
print([
    "reel_views_per_follower",
    "days_until_event",
    "campaign_progress",
    "rain_severity",
    "temperature_severity",
    "weather_severity",
    "overall_severity"
])

print("\nActual columns:")
print(df.columns.tolist())

Number of rows: 9
Number of columns: 24

Engineered variables:
['reel_views_per_follower', 'days_until_event', 'campaign_progress', 'rain_severity', 'temperature_severity', 'weather_severity', 'overall_severity']

Actual columns:
['date', 'tickets_sold', 'instagram_followers', 'reel_views', 'days_advertised', 'day_of_week', 'month', 'student_term_time', 'weather', 'rainfall', 'temperature', 'reel_views_per_follower', 'campaign_duration', 'days_until_event', 'campaign_progress', 'is_weekend', 'is_student_holiday', 'temperature_celsius', 'rainfall_mm', 'has_rain', 'rain_severity', 'temperature_severity', 'weather_severity', 'overall_severity']


In [28]:
df.to_csv("../data/JE001_real_improved.csv", index=False)